In [1]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('/kaggle/working/AKOrN/eval_sudoku.py'):
    !git clone https://github.com/DimitriBnvl/AKOrN.git
    os.chdir('/kaggle/working/AKOrN')
    !pip install -q -r requirements.txt
    !cd data && bash download_satnet.sh && bash download_rrn.sh
os.chdir('/kaggle/working/AKOrN')
import torch
print("GPUs:", torch.cuda.device_count(),
      "weights:", os.path.exists('/kaggle/input/datasets/dimitribnvl06/sudoku-akorn-final-weights/ema_99.pth'))

Cloning into 'AKOrN'...
remote: Enumerating objects: 324, done.
remote: Counting objects: 100% (120/120), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 324 (delta 85), reused 81 (delta 67), pack-reused 204 (from 2)
Receiving objects: 100% (324/324), 137.57 KiB | 2.59 MiB/s, done.
Resolving deltas: 100% (138/138), done.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 94.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.0/16.0 MB 81.0 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata

In [2]:
import pathlib, subprocess, os
os.chdir('/kaggle/working/AKOrN')
!git checkout -- eval_sudoku.py

p = pathlib.Path('eval_sudoku.py'); src = p.read_text()
anchor = "    for i, (X, Y, is_input) in tqdm.tqdm(enumerate(loader)):\n"
assert anchor in src, "anchor not found — stop and paste me the loop line"

src = src.replace(anchor,
    "    SHARD=int(os.environ.get('SHARD',0))\n"
    "    NSHARD=int(os.environ.get('NSHARD',1))\n"
    "    MAXB=int(os.environ.get('MAXB',0))\n"
    + anchor +
    "        if NSHARD>1 and (i%NSHARD)!=SHARD:\n            continue\n"
    "        if MAXB and totals >= MAXB:\n            break\n", 1)

src = src.replace('print(f"Vote accuracy: {accuracy_vote:.4f}")',
    'print(f"shard={SHARD}/{NSHARD} corrects_vote={corrects_vote} totals={totals} acc={accuracy_vote:.4f}")', 1)

p.write_text(src)
out = subprocess.run(['grep','-nE',
    "SHARD=int|NSHARD=int|MAXB=int|i%NSHARD|totals >= MAXB|corrects_vote=", 'eval_sudoku.py'],
    capture_output=True, text=True).stdout
print(out)
assert out.count('\n') >= 6, "expected 6 matches — patch incomplete, do not run"
print("OK — all edits present")

150:    SHARD=int(os.environ.get('SHARD',0))
151:    NSHARD=int(os.environ.get('NSHARD',1))
152:    MAXB=int(os.environ.get('MAXB',0))
154:        if NSHARD>1 and (i%NSHARD)!=SHARD:
156:        if MAXB and totals >= MAXB:
209:    print(f"shard={SHARD}/{NSHARD} corrects_vote={corrects_vote} totals={totals} acc={accuracy_vote:.4f}")

OK — all edits present


In [3]:
import subprocess, os, re, time
MP='/kaggle/input/datasets/dimitribnvl06/sudoku-akorn-final-weights/ema_99.pth'
NBOARDS  = 128
PER_CARD = NBOARDS // 2

def run_split(K, minimum_chunk, tag):
    args=['python','eval_sudoku.py','--data=ood','--model=akorn',
          f'--model_path={MP}','--T=128',f'--K={K}','--evote_type=sum',
          f'--batchsize={PER_CARD}']
    if minimum_chunk: args.append(f'--minimum_chunk={minimum_chunk}')
    procs, logs = [], []
    for shard in (0,1):
        log=f'/kaggle/working/{tag}_shard{shard}.txt'; logs.append(log)
        env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(shard),
                 SHARD=str(shard), NSHARD='2', MAXB=str(PER_CARD))
        procs.append(subprocess.Popen(args, cwd='/kaggle/working/AKOrN',
                     env=env, stdout=open(log,'w'), stderr=subprocess.STDOUT))
    time.sleep(45)
    os.system('nvidia-smi --query-gpu=index,utilization.gpu,memory.used --format=csv')
    for pr in procs: pr.wait()
    c=t=0
    for log in logs:
        m=re.search(r'corrects_vote=(\d+) totals=(\d+)', open(log).read())
        if m: c+=int(m[1]); t+=int(m[2])
    acc=c/t if t else 0.0
    print(f"[{tag}] K={K}: boards={t} corrects={c} acc={acc:.4f}")
    return K,acc,c,t

results=[run_split(100,None,'k100'), run_split(4096,256,'k4096')]
with open('/kaggle/working/curve_results.txt','w') as f:
    f.write(f"AKOrN Sudoku OOD — same first {NBOARDS} boards, T=128, evote=sum\n")
    for K,acc,c,t in results: f.write(f"K={K}: acc={acc:.4f} ({c}/{t})\n")
print("\n=== CURVE ===\n"+open('/kaggle/working/curve_results.txt').read())

index, utilization.gpu [%], memory.used [MiB]
0, 100 %, 2461 MiB
1, 100 %, 2461 MiB
[k100] K=100: boards=128 corrects=107 acc=0.8359
index, utilization.gpu [%], memory.used [MiB]
0, 0 %, 0 MiB
1, 0 %, 0 MiB
[k4096] K=4096: boards=0 corrects=0 acc=0.0000

=== CURVE ===
AKOrN Sudoku OOD — same first 128 boards, T=128, evote=sum
K=100: acc=0.8359 (107/128)
K=4096: acc=0.0000 (0/0)

